# 🌌 Stellar Classification — AutoGluon Baseline

**Goal:** establish a strong leaderboard baseline and verify that our local CV tracks the
public LB. Also verify which models and ensembling techniques perform better on this data (lightgbm vs catboost, stacking vs blending, etc).

Competition metric: **balanced accuracy**. We let AutoGluon bag + stack the full model
zoo, optimizing balanced accuracy directly, then report out-of-fold (OOF) balanced
accuracy and write a submission.

## 1. Install AutoGluon

In [1]:
# setuptools>=81 dropped pkg_resources, which AutoGluon's model import-guards still use
# (silently disabling XGBoost/CatBoost/etc). Pin it below 81, then install AutoGluon.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "setuptools<81", "autogluon.tabular[all]"], check=True)
import pkg_resources  # noqa: F401  -> raises if the pin didn't take
print("AutoGluon install OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.5 requires pyarrow>=21.0.0, but you have pyarrow 20.0.0 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
torchaudio 2.10.0+cpu requires torch==2.10.0, but you have to

AutoGluon install OK


## 2. Imports & data

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from autogluon.tabular import TabularPredictor

SEED = 42
TARGET = "class"

# Resolve data dir on Kaggle (/kaggle/input/<comp>) or locally (../data)
_kaggle = Path("/kaggle/input")
_hits = sorted(_kaggle.rglob("train.csv")) if _kaggle.exists() else []
DATA = _hits[0].parent if _hits else Path("../data")
print("Data dir:", DATA)

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print("train", train.shape, "test", test.shape)

Data dir: /kaggle/input/competitions/playground-series-s6e6
train (577347, 12) test (247435, 11)


## 3. Feature engineering — color indices

In [3]:
def add_colors(df):
    df = df.copy()
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]
    return df

NUM = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
COLOR = ["u_g", "g_r", "r_i", "i_z"]
CAT = ["spectral_type", "galaxy_population"]   # AutoGluon handles these natively
FEATS = NUM + COLOR + CAT

train = add_colors(train)
test = add_colors(test)
train_data = train[FEATS + [TARGET]]
test_data = test[FEATS]
CLASSES = sorted(train[TARGET].unique())   # ['GALAXY', 'QSO', 'STAR']
print("Classes:", CLASSES)

Classes: ['GALAXY', 'QSO', 'STAR']


## 4. Train AutoGluon (best_quality, balanced accuracy)

`dynamic_stacking=False` skips the DyStack sub-fit so the full time budget goes to the
real fit. Bump `time_limit` if you want a deeper search.

In [4]:
predictor = TabularPredictor(
    label=TARGET,
    eval_metric="balanced_accuracy",
    path="ag_models",
)
predictor.fit(
    train_data,
    presets="best_quality",
    time_limit=14400,  # 4h guardrail (well under the 12h session); stops early if done.
    dynamic_stacking=False,
)
predictor.leaderboard(silent=True)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       29.80 GB / 31.35 GB (95.1%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 14400s
AutoGluon will save models to "/kaggle/working/ag_models"
Train Data Rows:    577347
Train Data Columns: 14
Label Column:       class
AutoGluon infers your prediction problem is: 'multiclass' (because dtype of label-column == object).
	3 unique label values:  ['GALAXY', 'QSO', 'STAR']
	If 'multiclass' is not the correct pr

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBM_BAG_L2,0.956431,balanced_accuracy,3589.185486,9236.480448,23.200047,325.378508,2,True,10
1,WeightedEnsemble_L3,0.956431,balanced_accuracy,3589.334917,9288.510453,0.149431,52.030006,3,True,14
2,LightGBMXT_BAG_L2,0.955980,balanced_accuracy,3617.803140,9313.238336,51.817702,402.136396,2,True,9
3,NeuralNetFastAI_BAG_L2,0.955860,balanced_accuracy,3576.416957,10463.445919,10.431519,1552.343980,2,True,8
4,RandomForestEntr_BAG_L2,0.955792,balanced_accuracy,3599.570084,9882.649573,33.584645,971.547633,2,True,12
5,RandomForestGini_BAG_L2,0.955567,balanced_accuracy,3600.625166,10088.346807,34.639728,1177.244867,2,True,11
6,WeightedEnsemble_L2,0.955200,balanced_accuracy,3515.803112,7285.878656,0.148805,30.645632,2,True,7
7,CatBoost_BAG_L2,0.955188,balanced_accuracy,3566.655053,9074.275916,0.669615,163.173976,2,True,13
8,LightGBM_BAG_L1,0.954418,balanced_accuracy,210.063315,811.908545,210.063315,811.908545,1,True,3
9,LightGBMXT_BAG_L1,0.953850,balanced_accuracy,3294.961713,5027.770843,3294.961713,5027.770843,1,True,2


## 5. OOF balanced accuracy (our CV estimate to compare against the LB)

In [5]:
oof = predictor.predict_proba_oof()[CLASSES].to_numpy()
y_true = train[TARGET].to_numpy()

raw_oof_pred = np.array(CLASSES)[oof.argmax(1)]
raw_bal_acc = balanced_accuracy_score(y_true, raw_oof_pred)
print(f"OOF balanced accuracy (raw argmax) : {raw_bal_acc:.5f}   <- clean CV estimate")

OOF balanced accuracy (raw argmax) : 0.95643   <- clean CV estimate


## 6. Class-weight tuning for balanced accuracy

Plain argmax under-predicts the rare STAR/QSO classes; balanced accuracy punishes that.
We search per-class probability multipliers on the OOF predictions. Note the tuned OOF
score carries mild optimism (weights are fit on the same OOF), so treat the **raw** OOF
as the honest CV number and the LB as the arbiter.

In [6]:
def tune_class_weights(y_true, proba, classes, n_rounds=3):
    grid = np.linspace(0.2, 3.0, 29)
    w = np.ones(len(classes))
    score = lambda ww: balanced_accuracy_score(y_true, np.array(classes)[(proba * ww).argmax(1)])
    best = score(w)
    for _ in range(n_rounds):
        for k in range(len(w)):
            best_wk = w[k]
            for g in grid:
                w[k] = g
                s = score(w)
                if s > best:
                    best, best_wk = s, g
            w[k] = best_wk
    return w / w.mean(), best

weights, tuned_bal_acc = tune_class_weights(y_true, oof, CLASSES)
print(f"OOF balanced accuracy (tuned) : {tuned_bal_acc:.5f}  weights={dict(zip(CLASSES, weights.round(3)))}")

OOF balanced accuracy (tuned) : 0.96534  weights={'GALAXY': np.float64(0.333), 'QSO': np.float64(1.167), 'STAR': np.float64(1.5)}


## 7. Predict test & write submission

In [7]:
test_proba = predictor.predict_proba(test_data)[CLASSES].to_numpy()
test_pred = np.array(CLASSES)[(test_proba * weights).argmax(1)]

submission = pd.DataFrame({"id": test["id"], "class": test_pred})
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv")
print(submission["class"].value_counts(normalize=True).round(4))

np.save("oof_autogluon.npy", oof)
np.save("test_autogluon.npy", test_proba)
print("Wrote oof_autogluon.npy and test_autogluon.npy")


Wrote submission.csv
class
GALAXY    0.6290
QSO       0.2089
STAR      0.1621
Name: proportion, dtype: float64
Wrote oof_autogluon.npy and test_autogluon.npy
